In [0]:
-- ============================================
-- 🌎 VISTA DE KPIs POR UBICACIÓN
-- ============================================
-- Agrega métricas por ciudad/estado/país
-- Útil para análisis geográfico y mapas

CREATE OR REPLACE VIEW prueba_api.semantic.vw_kpis_by_location
COMMENT 'KPIs agregados por ubicación geográfica'
AS
SELECT 
    -- Identificadores geográficos
    location_sk,
    job_city,
    job_state,
    job_country,
    
    -- Métricas de volumen
    COUNT(*) AS total_ofertas,
    COUNT(DISTINCT employer_sk) AS empleadores_unicos,
    COUNT(DISTINCT employment_type_sk) AS tipos_empleo_disponibles,
    
    -- Métricas de modalidad
    SUM(CASE WHEN job_is_remote = true THEN 1 ELSE 0 END) AS ofertas_remotas,
    SUM(CASE WHEN job_is_remote = false THEN 1 ELSE 0 END) AS ofertas_presenciales,
    ROUND(AVG(CASE WHEN job_is_remote = true THEN 100.0 ELSE 0 END), 2) AS pct_remoto,
    
    -- Metadata
    MAX(fecha_carga) AS ultima_actualizacion
    
FROM prueba_api.gold.vw_market_overview
GROUP BY 
    location_sk,
    job_city,
    job_state,
    job_country;

## 📍 Ejemplos de Análisis Geográfico

Estos ejemplos muestran cómo usar la vista `vw_kpis_by_location` para analizar el mercado laboral por ubicación.

In [0]:
-- ============================================
-- EJEMPLO 1: TOP 10 ESTADOS CON MÁS OFERTAS
-- ============================================
-- Identifica los estados más activos en el mercado laboral

SELECT 
    job_state AS estado,
    job_country AS pais,
    total_ofertas,
    empleadores_unicos,
    tipos_empleo_disponibles,
    pct_remoto AS pct_trabajo_remoto,
    ofertas_remotas,
    ofertas_presenciales
FROM prueba_api.gold.vw_kpis_by_location
WHERE job_state IS NOT NULL
ORDER BY total_ofertas DESC
LIMIT 10;

In [0]:
-- ============================================
-- EJEMPLO 2: CIUDADES CON MÁS OFERTAS
-- ============================================
-- Encuentra las ciudades con mayor volumen de ofertas
-- Solo considera ciudades con al menos 5 ofertas

SELECT 
    job_city AS ciudad,
    job_state AS estado,
    job_country AS pais,
    total_ofertas,
    empleadores_unicos,
    tipos_empleo_disponibles,
    ofertas_remotas,
    pct_remoto AS pct_trabajo_remoto
FROM prueba_api.gold.vw_kpis_by_location
WHERE 
    job_city IS NOT NULL 
    AND total_ofertas >= 5  -- Solo ciudades con volumen significativo
ORDER BY total_ofertas DESC
LIMIT 10;

In [0]:
-- ============================================
-- EJEMPLO 3: RESUMEN POR PAÍS
-- ============================================
-- Agrega métricas a nivel de país para comparación internacional

SELECT 
    job_country AS pais,
    SUM(total_ofertas) AS total_ofertas,
    COUNT(DISTINCT job_state) AS estados_con_ofertas,
    COUNT(DISTINCT job_city) AS ciudades_con_ofertas,
    SUM(empleadores_unicos) AS total_empleadores,
    ROUND(AVG(pct_remoto), 2) AS pct_remoto_promedio,
    SUM(ofertas_remotas) AS total_ofertas_remotas,
    SUM(ofertas_presenciales) AS total_ofertas_presenciales
FROM prueba_api.gold.vw_kpis_by_location
WHERE job_country IS NOT NULL
GROUP BY job_country
ORDER BY total_ofertas DESC;

In [0]:
SELECT 
    job_country AS pais,
    SUM(total_ofertas) AS total_ofertas
FROM prueba_api.gold.vw_kpis_by_location
WHERE job_country IS NOT NULL
GROUP BY job_country
ORDER BY total_ofertas DESC;

In [0]:
-- ============================================
-- EJEMPLO 4: HUBS TECNOLÓGICOS
-- ============================================
-- Identifica ciudades con mayor diversidad de empleadores
-- Estas son probablemente los principales hubs tecnológicos

SELECT 
    job_city AS ciudad,
    job_state AS estado,
    job_country AS pais,
    empleadores_unicos,
    total_ofertas,
    ROUND(total_ofertas * 1.0 / empleadores_unicos, 2) AS ofertas_por_empleador,
    tipos_empleo_disponibles,
    pct_remoto AS pct_trabajo_remoto
FROM prueba_api.gold.vw_kpis_by_location
WHERE 
    job_city IS NOT NULL 
    AND empleadores_unicos >= 10  -- Ciudades con al menos 10 empleadores diferentes
ORDER BY empleadores_unicos DESC, total_ofertas DESC
LIMIT 15;

In [0]:
-- ============================================
-- EJEMPLO 5: UBICACIONES CON MAYOR FLEXIBILIDAD
-- ============================================
-- Encuentra ubicaciones donde el trabajo remoto es más común

SELECT 
    job_city AS ciudad,
    job_state AS estado,
    job_country AS pais,
    total_ofertas,
    ofertas_remotas,
    pct_remoto AS pct_trabajo_remoto,
    empleadores_unicos
FROM prueba_api.gold.vw_kpis_by_location
WHERE 
    job_city IS NOT NULL 
    AND total_ofertas >= 3  -- Al menos 3 ofertas para tener datos significativos
    AND pct_remoto > 0  -- Solo ubicaciones con al menos una oferta remota
ORDER BY pct_remoto DESC, total_ofertas DESC
LIMIT 15;

## 📊 Insights del Análisis Geográfico

A partir de los ejemplos anteriores, podemos observar:

### 🌎 Distribución Global
* **Estados Unidos** lidera con 91 ofertas
* **India** (88 ofertas), **Colombia** (88 ofertas) y **Brasil** (82 ofertas) tienen alto volumen
* **Chile** con 69 ofertas
* El mercado **REMOTE** (trabajo 100% remoto) representa una porción significativa

### 🏙️ Principales Mercados por Estado/Región
* **Región Metropolitana (Chile)**: 54 ofertas, 37 empleadores - principal hub latinoamericano
* **Bogotá (Colombia)**: 45 ofertas, 35 empleadores
* **São Paulo (Brasil)**: 23 ofertas, 18 empleadores
* **Karnataka (India)**: 13 ofertas (hub tecnológico de Bangalore)

### 🏠 Trabajo Remoto
* **Brasil Remote**: 10 ofertas 100% remotas
* La mayoría de países latinoamericanos tienen **0% de trabajo remoto**
* El mercado USA es mayormente **presencial** con pocas excepciones

